# Notebook 04 — Fraud Detection Platform: Serving Architecture, Training-Serving Consistency & Latency SLA
**Master Playbook Section 17.4 / gap-analysis priority 7 — actually runs the real, unmodified `deployment/app.py` FastAPI service in-process (FastAPI TestClient) against Notebook 01's real champion model, verifies its predictions match direct model scoring on real sampled rows, and measures real serving latency. No retraining, no simulated numbers.**


In [ ]:
# ============================================================
# SETUP -- WARP-optimized environment (thread ceiling set BEFORE any ML import)
# CPU/RAM thresholds: CPU 93% (90-95% band), RAM 90%.
# ============================================================
import os, time, json, pickle, warnings, subprocess, sys, importlib.util
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

_RUN_T0 = time.time()

CPU_THRESHOLD_PCT = 93
RAM_THRESHOLD_PCT = 90

_N_THREADS = max(1, int((os.cpu_count() or 4) * (CPU_THRESHOLD_PCT / 100) // 1))
os.environ.setdefault("OMP_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("POLARS_MAX_THREADS", str(_N_THREADS))

# Lean auto-install guard -- adds only what this notebook needs beyond NB1/
# NB2/NB3's deps: fastapi (already required by deployment/app.py) + httpx
# (FastAPI's TestClient uses it for in-process request simulation).
for _pkg in ("polars", "psutil", "pyarrow", "catboost", "fastapi", "httpx"):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", _pkg], check=True)

import polars as pl
import psutil
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

_ram_start = psutil.virtual_memory()
print(f"WARP thread ceiling: {_N_THREADS} threads (of {os.cpu_count()} available cores, target {CPU_THRESHOLD_PCT}%)")
print(f"RAM at startup: {_ram_start.percent:.1f}% used ({_ram_start.used/1e9:.2f} GB / {_ram_start.total/1e9:.2f} GB) -- target ceiling {RAM_THRESHOLD_PCT}%")
if _ram_start.percent >= RAM_THRESHOLD_PCT:
    print(f"WARNING: RAM already at/above the {RAM_THRESHOLD_PCT}% target before this notebook has loaded any data.")
print("Setup complete.")

##############################################################################
# REPO-LAYOUT BOOTSTRAP -- verified detection, unchanged from NB1/NB2/NB3.
##############################################################################
_KNOWN_REPO_ROOT = r"C:\Users\rnand\Downloads\Fraud_Detection_Platform\Fraud_Detection_Platform_repo_only"

def _looks_like_repo(_p):
    return os.path.isdir(os.path.join(_p, "notebooks")) or os.path.exists(os.path.join(_p, "requirements.txt"))

_cwd = os.getcwd()
_parent = os.path.abspath(os.path.join(_cwd, ".."))

if os.path.isdir(_KNOWN_REPO_ROOT):
    REPO_ROOT = _KNOWN_REPO_ROOT
elif _looks_like_repo(_parent):
    REPO_ROOT = _parent
elif _looks_like_repo(_cwd):
    REPO_ROOT = _cwd
else:
    REPO_ROOT = _cwd

_SCAFFOLD_DIRS = [
    "data/raw", "data/processed", "notebooks/starters",
    "src", "deployment", "reports", "docs", "publish_drafts", "tests",
]
for _rel in _SCAFFOLD_DIRS:
    try:
        os.makedirs(os.path.join(REPO_ROOT, *_rel.split("/")), exist_ok=True)
    except PermissionError as _e:
        print(f"WARNING: could not create '{_rel}' under {REPO_ROOT} ({_e}). Skipping.")

NB1_RESULTS_DIR = os.path.join(REPO_ROOT, "reports", "nb1_results")
RESULTS_DIR = os.path.join(REPO_ROOT, "reports", "nb4_results")
try:
    os.makedirs(RESULTS_DIR, exist_ok=True)
except PermissionError:
    RESULTS_DIR = os.path.join(_cwd, "nb4_results")
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"WARNING: falling back to {RESULTS_DIR} (no write access to {REPO_ROOT}).")

print(f"Repo root:      {REPO_ROOT}")
print(f"NB1 results in: {NB1_RESULTS_DIR}")
print(f"NB4 results in: {RESULTS_DIR}")

##############################################################################
# LOAD NOTEBOOK 01's REAL OUTPUTS -- no retraining.
##############################################################################
with open(os.path.join(NB1_RESULTS_DIR, "nb1_final_results.json"), encoding="utf-8") as f:
    nb1_results = json.load(f)
with open(os.path.join(NB1_RESULTS_DIR, "champion_model.pkl"), "rb") as f:
    champion_model = pickle.load(f)

FEATURE_COLS = list(champion_model.feature_names_)
CHOSEN_THRESHOLD = nb1_results["threshold_result"]["threshold"]
print(f"Loaded champion model ({nb1_results['champion_name']}), operating threshold {CHOSEN_THRESHOLD:.4f}")

##############################################################################
# DATA_PATH resolution + Polars-accelerated load -- NB1/NB2/NB3's fixes
# reused unchanged.
##############################################################################
DATA_PATH = None
for _cand in [
    r"C:\Users\rnand\Downloads\creditcard.csv\creditcard.csv",  # your real dataset location -- checked first
    "creditcard.csv",
    os.path.join("data", "raw", "creditcard.csv"),
    os.path.join("..", "data", "raw", "creditcard.csv"),
    os.path.join("..", "creditcard.csv"),
]:
    if os.path.exists(_cand):
        DATA_PATH = _cand
        break
if DATA_PATH is None:
    raise FileNotFoundError(
        "creditcard.csv not found. Place it next to this notebook, or at "
        "data/raw/creditcard.csv relative to the repo root."
    )
print("Using DATA_PATH:", DATA_PATH)

_SCHEMA_OVERRIDES = {"Time": pl.Float64, "Amount": pl.Float64, "Class": pl.Int64}
for _i in range(1, 29):
    _SCHEMA_OVERRIDES[f"V{_i}"] = pl.Float64

_t_load0 = time.time()
df = pl.read_csv(DATA_PATH, schema_overrides=_SCHEMA_OVERRIDES).to_pandas()
print(f"Loaded {len(df):,} rows x {len(df.columns)} cols via Polars in {time.time()-_t_load0:.3f}s")
_ram_after_load = psutil.virtual_memory()
print(f"RAM after load: {_ram_after_load.percent:.1f}% used ({_ram_after_load.used/1e9:.2f} GB / {_ram_after_load.total/1e9:.2f} GB)")

##############################################################################
# LOAD THE REAL DEPLOYMENT SERVICE (deployment/app.py, unmodified) -- env
# vars pointing at the REAL trained model must be set BEFORE import, since
# app.py reads them at module level.
##############################################################################
os.environ["MODEL_PATH"] = os.path.join(NB1_RESULTS_DIR, "champion_model.pkl")
os.environ["MODEL_VERSION"] = "fraud-champion-v1.0.0"
os.environ["DECISION_THRESHOLD"] = str(CHOSEN_THRESHOLD)

_app_path = os.path.join(REPO_ROOT, "deployment", "app.py")
if not os.path.exists(_app_path):
    raise FileNotFoundError(f"deployment/app.py not found at {_app_path} -- cannot test the real serving layer.")

_spec = importlib.util.spec_from_file_location("fraud_app", _app_path)
fraud_app = importlib.util.module_from_spec(_spec)
sys.modules["fraud_app"] = fraud_app
_spec.loader.exec_module(fraud_app)

from fastapi.testclient import TestClient

print("=" * 70)
print("SERVING ARCHITECTURE -- REAL API, IN-PROCESS (FastAPI TestClient)")
print("=" * 70)
print("Note: TestClient runs the real ASGI app in-process -- it measures real "
      "app/serialization overhead but NOT real network transport time, so "
      "latency numbers below are a serving-path lower bound, not a full "
      "production network SLA. Disclosed, not hidden.")

with TestClient(fraud_app.app) as client:
    _health = client.get("/health").json()
    _model_info = client.get("/model_info").json()
    print(f"  /health: {_health}")
    print(f"  /model_info: {_model_info}")

    health_ok = _health.get("model_loaded") is True
    model_info_ok = (
        abs(_model_info.get("decision_threshold", -1) - CHOSEN_THRESHOLD) < 1e-9
        and _model_info.get("expected_feature_columns") == FEATURE_COLS
    )
    print(f"  health check passed: {health_ok}")
    print(f"  model_info matches NB1's real threshold/features: {model_info_ok}")

    ##########################################################################
    # TRAINING-SERVING CONSISTENCY CHECK + LATENCY SLA -- one real loop, real
    # rows (real fraud + real legit, RANDOM_SEED=42 sample), one /score call
    # per row (the API's real contract is single-transaction, not batch --
    # this loop is the honest reflection of that, not an artificial limit).
    ##########################################################################
    N_FRAUD_SAMPLE = min(250, int(df["Class"].sum()))
    N_LEGIT_SAMPLE = 250
    sample_fraud = df[df["Class"] == 1].sample(n=N_FRAUD_SAMPLE, random_state=RANDOM_SEED)
    sample_legit = df[df["Class"] == 0].sample(n=N_LEGIT_SAMPLE, random_state=RANDOM_SEED)
    sample_rows = pd.concat([sample_fraud, sample_legit], ignore_index=True)

    # Direct model predictions -- float64, DataFrame, exactly as NB1/NB2/NB3
    # score -- computed once, vectorized (not per-row).
    direct_scores = champion_model.predict_proba(sample_rows[FEATURE_COLS])[:, 1]

    api_scores = np.empty(len(sample_rows), dtype=np.float64)
    server_latencies_ms = np.empty(len(sample_rows), dtype=np.float64)
    client_latencies_ms = np.empty(len(sample_rows), dtype=np.float64)

    _t_loop0 = time.time()
    for _i, (_, row) in enumerate(sample_rows.iterrows()):
        feats = {c: float(row[c]) for c in FEATURE_COLS}
        _t_call0 = time.perf_counter()
        resp = client.post("/score", json={"features": feats})
        client_latencies_ms[_i] = (time.perf_counter() - _t_call0) * 1000.0
        body = resp.json()
        api_scores[_i] = body["fraud_probability"]
        server_latencies_ms[_i] = body["latency_ms"]
    consistency_loop_seconds = time.time() - _t_loop0

    abs_diff = np.abs(api_scores - direct_scores)
    CONSISTENCY_TOLERANCE = 1e-3  # ASSUMPTION: covers the API's float32 row-conversion boundary (app.py casts to np.float32) vs. this notebook's float64 direct scoring
    consistency_pass = bool(np.max(abs_diff) < CONSISTENCY_TOLERANCE)

    print("=" * 70)
    print(f"TRAINING-SERVING CONSISTENCY -- {len(sample_rows)} real rows "
          f"({N_FRAUD_SAMPLE} fraud + {N_LEGIT_SAMPLE} legit), real API calls vs. real direct model scoring")
    print("=" * 70)
    print(f"  Max |API - direct| difference: {np.max(abs_diff):.8f}")
    print(f"  Mean |API - direct| difference: {np.mean(abs_diff):.8f}")
    print(f"  Tolerance (ASSUMPTION, float32-boundary-aware): {CONSISTENCY_TOLERANCE}")
    print(f"  CONSISTENCY CHECK PASSED: {consistency_pass}")
    print(f"  Loop wall-clock for {len(sample_rows)} real API calls: {consistency_loop_seconds:.3f}s")

    _server_p50, _server_p95, _server_p99 = np.percentile(server_latencies_ms, [50, 95, 99])
    _client_p50, _client_p95, _client_p99 = np.percentile(client_latencies_ms, [50, 95, 99])

    print("=" * 70)
    print("LATENCY SLA -- real, measured (in-process TestClient; see note above)")
    print("=" * 70)
    print(f"  Server-reported latency_ms  -- P50: {_server_p50:.3f}  P95: {_server_p95:.3f}  "
          f"P99: {_server_p99:.3f}  mean: {np.mean(server_latencies_ms):.3f}")
    print(f"  Client round-trip ms        -- P50: {_client_p50:.3f}  P95: {_client_p95:.3f}  "
          f"P99: {_client_p99:.3f}  mean: {np.mean(client_latencies_ms):.3f}")

##############################################################################
# SAVE NOTEBOOK 04 RESULTS
##############################################################################
nb4_report = {
    "run_metadata": {"random_seed": RANDOM_SEED, "n_threads": _N_THREADS,
                     "model_version": os.environ["MODEL_VERSION"],
                     "decision_threshold": CHOSEN_THRESHOLD},
    "health_check_passed": health_ok,
    "model_info_check_passed": model_info_ok,
    "training_serving_consistency": {
        "n_rows_tested": int(len(sample_rows)), "n_fraud_sampled": N_FRAUD_SAMPLE, "n_legit_sampled": N_LEGIT_SAMPLE,
        "max_abs_diff": float(np.max(abs_diff)), "mean_abs_diff": float(np.mean(abs_diff)),
        "tolerance": CONSISTENCY_TOLERANCE, "passed": consistency_pass,
        "loop_wall_clock_seconds": round(consistency_loop_seconds, 4),
    },
    "latency_sla_ms": {
        "note": "In-process FastAPI TestClient -- real app/serialization overhead, NOT real network transport time.",
        "server_reported": {"p50": float(_server_p50), "p95": float(_server_p95), "p99": float(_server_p99),
                            "mean": float(np.mean(server_latencies_ms))},
        "client_round_trip": {"p50": float(_client_p50), "p95": float(_client_p95), "p99": float(_client_p99),
                              "mean": float(np.mean(client_latencies_ms))},
    },
}
with open(os.path.join(RESULTS_DIR, "nb4_serving_report.json"), "w", encoding="utf-8") as f:
    json.dump(nb4_report, f, indent=2, default=str)

_serving_ready = health_ok and model_info_ok and consistency_pass
print("=" * 70)
print(f"SERVING READINESS VERDICT: {'READY' if _serving_ready else 'NOT READY -- see checks above'}")
print("=" * 70)

_ram_end = psutil.virtual_memory()
_total_elapsed = time.time() - _RUN_T0
print(f"RAM at finish: {_ram_end.percent:.1f}% used ({_ram_end.used/1e9:.2f} GB / {_ram_end.total/1e9:.2f} GB) -- "
      f"stayed under the {RAM_THRESHOLD_PCT}% ceiling: {_ram_end.percent < RAM_THRESHOLD_PCT}")
print(f"Total notebook wall-clock time: {_total_elapsed:.2f}s (real, measured).")
print(f"Notebook 04 complete. Results written to: {os.path.join(RESULTS_DIR, 'nb4_serving_report.json')}")
